# Домашнее задание: харнесс и бенчмарки (MERA / MultiQ)

План:
1. Загружаем `MultiQ` из MERA (train), берём 25 **случайных** примеров, собираем готовые промпты из `instruction`.
2. Пишем метрики: **Exact Match** и **F1 по леммам**, считаем на двух бейзлайнах (частотный биграмм + неочевидный эвристический бейзлайн «мост → предложение → сущность»).
3. Эксперимент 1: `Qwen/Qwen3.5-0.8B` без thinking.
4. Эксперимент 2: тот же сетап + system prompt, написанный по разбору ошибок эксперимента 1.
5. Эксперимент 3: thinking с бюджетом на размышления, двухэтапной генерацией и постобработкой.

Ноутбук рассчитан на GPU (Colab T4 хватает), на CPU тоже отработает, но медленно.

In [ ]:
%pip install -q -U transformers accelerate datasets pymorphy3 razdel pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 57.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pan

In [ ]:
import re
import time
import random
import string
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", 120)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## Данные

Берём train split, перемешиваем с фиксированным сидом и берём 25 примеров — так выборка случайная, но воспроизводимая.

In [ ]:
try:
    ds = load_dataset("MERA-evaluation/MERA", "multiq", split="train")
except Exception as e:  # в новых версиях datasets скрипты загрузки не поддерживаются - берем parquet-версию
    print("Обычная загрузка не сработала:", repr(e)[:200])
    ds = load_dataset("MERA-evaluation/MERA", "multiq", split="train", revision="refs/convert/parquet")

print(ds)
ds[0]

README.md:   0%|          | 0.00/168k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/2.87M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/3.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1056 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'inputs', 'outputs', 'meta'],
    num_rows: 1056
})


{'instruction': 'Даны два текста:\nТекст 1: {support_text}\nТекст 2: {text}\nОпираясь на данные тексты, ответьте на вопрос: {question}\nВаш ответ не должен содержать дополнительные объяснения.\nОтвет:',
 'inputs': {'text': 'Нижний Новгород (в разговорной речи часто — "Нижний", c XIII по XVII век — Новгород Низовской земли, с 7 октября 1932 по 22 октября 1990 года — Горький) — город в центральной России, административный центр Приволжского федерального округа и Нижегородской области. Второй по численности населения город в Приволжском федеральном округе и на реке Волге.\\n\\nКультура.\\nИсторический центр Нижнего Новгорода, расположенный в Нагорной части города, несмотря на значительные перестройки, сохранил значительное число исторических гражданских строений XVIII — начала XX веков, включая многочисленные памятники деревянного зодчества. Дмитриевская башня Кремля выходит на историческую площадь Минина и Пожарского. Нижегородский кремль является официальной резиденцией Городской думы Н

In [ ]:
N_EXAMPLES = 25
sample = ds.shuffle(seed=SEED).select(range(N_EXAMPLES))
print(sample["meta"][:5])

[{'id': 129, 'bridge_answers': 'Вятки'}, {'id': 833, 'bridge_answers': 'Сирия'}, {'id': 243, 'bridge_answers': 'Алтая'}, {'id': 584, 'bridge_answers': 'Суходрев'}, {'id': 548, 'bridge_answers': 'Ереван'}]


### Сборка промптов

`instruction` содержит плейсхолдеры `{support_text}`, `{text}`, `{question}`, значения лежат в словаре `inputs`.
Подставляем через регулярку, а не через `str.format`: в текстах Википедии могут встретиться фигурные скобки, и `format` на них упадёт.
Заодно превращаем экранированные `\\n` из текстов в настоящие переносы строк.

In [ ]:
PLACEHOLDER = re.compile(r"\{(\w+)\}")


def clean_field(value):
    return str(value).replace("\\n", "\n").strip()


def fill_instruction(instruction, inputs):
    def repl(m):
        key = m.group(1)
        return clean_field(inputs[key]) if key in inputs else m.group(0)
    return PLACEHOLDER.sub(repl, instruction).strip()


def gold_answers(outputs):
    # в MultiQ несколько допустимых ответов перечисляются через ";" (так же делает официальный скорер MERA)
    return [a.strip() for a in str(outputs).split(";") if a.strip()]


records = []
for ex in sample:
    inputs = ex["inputs"]
    records.append({
        "id": ex["meta"]["id"],
        "prompt": fill_instruction(ex["instruction"], inputs),
        "question": clean_field(inputs["question"]),
        "support_text": clean_field(inputs["support_text"]),
        "text": clean_field(inputs["text"]),
        "gold": gold_answers(ex["outputs"]),
        "bridge": ex["meta"].get("bridge_answers", ""),
    })

data = pd.DataFrame(records)
assert not data["prompt"].str.contains(r"\{(?:support_text|text|question)\}").any(), "остались незаполненные плейсхолдеры"
print(data.loc[0, "prompt"][:1500])
data[["id", "question", "gold", "bridge"]]

Найдите в приведенных двух текстах ответ на заданный вопрос и выпишите этот ответ. Он должен являться словом или короткой фразой, дополнительные объяснения приводить не нужно.
Текст 1: Мурдюг (Большой Мурдюг) — река в России, протекает в Оричевском районе Кировской области. Устье реки находится в 494 км по левому берегу реки Вятки. Длина реки составляет 21 км. В 1,8 километрах от устья принимает слева реку Тумана.

Данные водного реестра.
По данным государственного водного реестра России относится к Камскому бассейновому округу, водохозяйственный участок реки — Вятка от города Котельнич до водомерного поста посёлка городского типа Аркуль, речной подбассейн реки — Вятка. Речной бассейн реки — Кама.
Текст 2: Вятка ( , , ) — река в Европейской части России, крупнейший правый приток Камы (бассейн Волги) и второй по величине после Белой. Длина реки — 1314 км, площадь водосборного бассейна — 129 000 кв км.

География.
Схема бассейна Вятки. Начинается на Верхнекамской возвышенности на севере 

,id,question,gold,bridge
0,129,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки
1,833,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия
2,243,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Алтая
3,584,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев
4,548,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван
5,807,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма
6,47,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро
7,218,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Генрих
8,128,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия
9,321,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск


## Метрики (2 балла)

**Базовая предобработка** (для обеих метрик): нижний регистр, `ё → е`, удаление пунктуации (все символы юникод-категории `P*`, т.е. и «ёлочки», и тире), схлопывание пробелов.

1. **Exact Match** — совпадение нормализованных строк.
2. **F1 по леммам** — нормализованный текст токенизируем (`razdel`), лемматизируем (`pymorphy3`) и считаем F1 по мультимножествам лемм:
   - precision = доля лемм предсказания, которые есть в правильном ответе;
   - recall = доля лемм правильного ответа, которые есть в предсказании;
   - F1 = 2PR / (P + R).

Лемматизация важна именно для MultiQ: золотые ответы часто стоят в той форме, в которой встретились в тексте («Городской **думы**»), а модель отвечает в начальной форме («Городская **дума**») — это верный ответ, и EM его не засчитает, а F1 по леммам засчитает.

Если правильных ответов несколько, берём максимум по ним (как в официальном скорере MERA).

In [ ]:
import pymorphy3
from razdel import tokenize as razdel_tokenize

morph = pymorphy3.MorphAnalyzer()


def normalize(text):
    text = str(text).lower().replace("ё", "е")
    text = "".join(" " if unicodedata.category(ch).startswith("P") else ch for ch in text)
    return " ".join(text.split())


_lemma_cache = {}


def lemmatize(word):
    if word not in _lemma_cache:
        _lemma_cache[word] = morph.parse(word)[0].normal_form.replace("ё", "е")
    return _lemma_cache[word]


def to_lemmas(text):
    return [lemmatize(t.text) for t in razdel_tokenize(normalize(text))]


def exact_match(pred, golds):
    pred_n = normalize(pred)
    return float(any(pred_n == normalize(g) for g in golds))


def lemma_f1_single(pred, gold):
    pred_l, gold_l = to_lemmas(pred), to_lemmas(gold)
    if not pred_l or not gold_l:
        return float(pred_l == gold_l)
    common = sum((Counter(pred_l) & Counter(gold_l)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_l)
    recall = common / len(gold_l)
    return 2 * precision * recall / (precision + recall)


def lemma_f1(pred, golds):
    return max(lemma_f1_single(pred, g) for g in golds)


def evaluate(preds, golds_list=None):
    golds_list = data["gold"].tolist() if golds_list is None else golds_list
    em = [exact_match(p, g) for p, g in zip(preds, golds_list)]
    f1 = [lemma_f1(p, g) for p, g in zip(preds, golds_list)]
    return {"EM": float(np.mean(em)), "F1_lemma": float(np.mean(f1))}, em, f1

In [ ]:
# санити-чеки метрик
assert exact_match("Городской думы.", ["городской Думы"]) == 1.0
assert exact_match("Городская дума", ["Городской думы"]) == 0.0
assert lemma_f1("Городская дума", ["Городской думы"]) == 1.0   # формы разные, леммы те же
assert lemma_f1("Ответ: Городская дума", ["Городской думы"]) == 0.8  # P=2/3, R=1
assert lemma_f1("Волга", ["Ока", "Волга"]) == 1.0                # максимум по нескольким ответам
assert lemma_f1("", ["Волга"]) == 0.0
print(to_lemmas("Городской думы Нижнего Новгорода, 1932 г."))
print("ok")

['городской', 'дума', 'нижний', 'новгород', '1932', 'г']
ok


## Бейзлайны

**Бейзлайн 1 (из подсказки)** — самый частотный биграмм промпта. Чтобы он был хоть немного осмысленным, считаем биграммы только по текстам (без текста инструкции) и выкидываем биграммы, в которых есть служебные слова.

**Бейзлайн 2 (неочевидный, +0.5)** — эвристика, повторяющая устройство multi-hop вопроса:
1. Вопрос в MultiQ устроен как «*Как называется X города, где родился Y?*»: в Тексте 1 находится мост (город), в Тексте 2 — ответ про этот мост.
2. Разбиваем Текст 2 на предложения и выбираем то, у которого больше всего общих лемм с вопросом (знаменательные слова вопроса — «законодательный орган», «река», «год» и т.п. — обычно перефразированы рядом с ответом).
3. В этом предложении выделяем кандидатов: цепочки слов с заглавной буквы (кавычки и дефисы допускаются) и числа. Если вопрос про год/дату/количество — берём числа.
4. Выкидываем кандидатов, чьи леммы целиком содержатся в вопросе, в Тексте 1 или в заголовочной сущности Текста 2 (это либо сам мост — Текст 2 и есть статья о нём, — либо сущность из вопроса), и берём ближайшего к словам вопроса.

In [ ]:
STOP_POS = {"PREP", "CONJ", "PRCL", "NPRO", "INTJ"}
STOPWORDS = {"это", "быть", "который", "также", "свой", "весь", "такой", "этот", "тот", "год",
             # слова-связки multi-hop вопросов, которые указывают на мост, а не на ответ
             "как", "какой", "называться", "являться", "город", "родиться", "умереть", "находиться", "где"}


def content_lemmas(text):
    out = []
    for t in razdel_tokenize(normalize(text)):
        p = morph.parse(t.text)[0]
        if p.tag.POS in STOP_POS or p.normal_form in STOPWORDS:
            continue
        out.append(p.normal_form.replace("ё", "е"))
    return out


def baseline_bigram(row):
    lemmas_text = f"{row['support_text']} {row['text']}"
    tokens = [t.text for t in razdel_tokenize(lemmas_text) if any(ch.isalnum() for ch in t.text)]
    bigrams = Counter()
    for a, b in zip(tokens, tokens[1:]):
        pa, pb = morph.parse(a.lower())[0], morph.parse(b.lower())[0]
        if pa.tag.POS in STOP_POS or pb.tag.POS in STOP_POS:
            continue
        bigrams[(a, b)] += 1
    if not bigrams:
        return ""
    return " ".join(bigrams.most_common(1)[0][0])


bigram_preds = [baseline_bigram(r) for _, r in data.iterrows()]
bigram_scores, *_ = evaluate(bigram_preds)
bigram_scores

{'EM': 0.04, 'F1_lemma': 0.04}

In [ ]:
from razdel import sentenize

NUMERIC_Q = re.compile(r"\b(в каком году|когда|сколько|какого числа|в каком веке|в какой год)\b", re.I)
WORD = r"[«\"]?[A-ZА-ЯЁ][\w\-\.’']*[»\"]?"
CAP_SPAN = re.compile(rf"{WORD}(?:\s+(?:{WORD}|де|фон|ван|ла|и|на|-на-)\s*)*")
NUM_SPAN = re.compile(r"\b\d{1,4}(?:[–-]\d{1,4})?\b")


def heuristic_baseline(row):
    q_lemmas = set(content_lemmas(row["question"]))
    s1_lemmas = set(to_lemmas(row["support_text"]))
    # Текст 2 - статья о мосте, ее первая сущность - сам мост (часто под другим именем, чем в Тексте 1)
    title = CAP_SPAN.match(row["text"].lstrip(" «\""))
    title_lemmas = set(to_lemmas(title.group(0))) if title else set()
    sentences = [s.text for s in sentenize(row["text"]) if s.text.strip()]
    if not sentences:
        return ""

    # 1. предложение текста 2, ближайшее к вопросу
    def score(sent):
        return len(q_lemmas & set(content_lemmas(sent)))
    best = max(sentences, key=score)

    # 2. кандидаты
    if NUMERIC_Q.search(row["question"]):
        cands = [m for m in NUM_SPAN.finditer(best)]
    else:
        cands = []
        for m in CAP_SPAN.finditer(best):
            span = m.group(0).strip(" «»\".,;:").strip()
            if not span:
                continue
            lem = set(to_lemmas(span))
            # выкидываем мост/сущности из вопроса и текста 1
            if lem and (lem <= q_lemmas or lem <= s1_lemmas or lem <= title_lemmas):
                continue
            # первое слово предложения с заглавной буквы - обычно не сущность, если это не имя собственное
            if m.start() == 0 and len(span.split()) == 1:
                p = morph.parse(span.lower())[0]
                if "Name" not in p.tag and "Geox" not in p.tag and "Orgn" not in p.tag:
                    continue
            cands.append(m)
    if not cands:
        return ""

    # 3. ближайший к словам вопроса кандидат
    anchors = [m.start() for m in re.finditer(r"\w+", best)
               if lemmatize(normalize(m.group(0))) in q_lemmas]
    def dist(m):
        return min((abs(m.start() - a) for a in anchors), default=m.start())
    return min(cands, key=dist).group(0).strip(" «»\".,;:").strip()


heuristic_preds = [heuristic_baseline(r) for _, r in data.iterrows()]
heuristic_scores, h_em, h_f1 = evaluate(heuristic_preds)
print("bigram   :", bigram_scores)
print("heuristic:", heuristic_scores)
pd.DataFrame({"question": data["question"], "gold": data["gold"], "pred": heuristic_preds, "F1": h_f1}).head(15)

bigram   : {'EM': 0.04, 'F1_lemma': 0.04}
heuristic: {'EM': 0.2, 'F1_lemma': 0.2}


,question,gold,pred,F1
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Европейской,0.0
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сурийя,0.0
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Туулу Алтай,0.0
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],,0.0
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],,0.0
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Светишонка,0.0
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Лиге МХ,1.0
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Францию,0.0
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сурийя,0.0
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Евгению Тефтелеву,1.0


## Модель

`Qwen/Qwen3.5-0.8B`: маленькая, инструктивная, поддерживает thinking через `enable_thinking` в чат-темплейте.
Генерация везде жадная (`do_sample=False`), чтобы сравнение экспериментов не зависело от случайности сэмплирования.
Для скорости генерируем батчами с левым паддингом.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3.5-0.8B"   # запасной вариант: "Qwen/Qwen3-0.6B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if device == "cuda" else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
except (ValueError, KeyError) as e:
    # Qwen3.5 - мультимодальная архитектура; если CausalLM-обертка не подхватилась, грузим полную модель
    print("AutoModelForCausalLM не подошел:", repr(e)[:200])
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
model.eval()

BATCH_SIZE = 8 if device == "cuda" else 1
print(MODEL_NAME, device, dtype)

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen/Qwen3.5-0.8B cuda torch.bfloat16


In [ ]:
KEEP_TOKENS = {"<think>", "</think>"}
SPECIAL_TO_STRIP = [t for t in tokenizer.all_special_tokens if t not in KEEP_TOKENS]


def build_prompt(user_prompt, system_prompt=None, enable_thinking=False):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )


def decode_keep_think(token_ids):
    # skip_special_tokens=True мог бы съесть теги размышлений, поэтому чистим служебные токены вручную
    text = tokenizer.decode(token_ids, skip_special_tokens=False)
    for tok in SPECIAL_TO_STRIP:
        text = text.replace(tok, "")
    return text


@torch.no_grad()
def generate(texts, max_new_tokens, stop_strings=None, batch_size=BATCH_SIZE, desc="", gen_kwargs=None):
    # возвращает сгенерированные продолжения и число сгенерированных токенов
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))  # похожие по длине - в один батч
    outs, n_tokens = [None] * len(texts), [0] * len(texts)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        batch = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True,
                          add_special_tokens=False).to(model.device)
        kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None,
                      pad_token_id=tokenizer.pad_token_id)
        if stop_strings:
            kwargs.update(stop_strings=stop_strings, tokenizer=tokenizer)
        if gen_kwargs:
            kwargs.update(gen_kwargs)   # например, параметры сэмплирования для thinking
        out = model.generate(**batch, **kwargs)
        gen = out[:, batch["input_ids"].shape[1]:]
        for i, g in zip(idx, gen):
            outs[i] = decode_keep_think(g)
            n_tokens[i] = int((g != tokenizer.pad_token_id).sum())
        print(f"\r{desc} {min(start + batch_size, len(order))}/{len(order)}", end="")
    print()
    return outs, n_tokens

In [ ]:
# как выглядит промпт после чат-темплейта (без thinking)
print(build_prompt("Пример вопроса", enable_thinking=False))
print("-" * 40)
print(build_prompt("Пример вопроса", enable_thinking=True))
print("Токенов в самом длинном промпте:", max(len(tokenizer(build_prompt(p))["input_ids"]) for p in data["prompt"]))

<|im_start|>user
Пример вопроса<|im_end|>
<|im_start|>assistant
<think>

</think>


----------------------------------------
<|im_start|>user
Пример вопроса<|im_end|>
<|im_start|>assistant
<think>

Токенов в самом длинном промпте: 863


### Постобработка ответа

Одна функция для всех экспериментов (чтобы сравнение было честным):
- выкидываем блок размышлений: всё до последнего `</think>` (если тег не закрылся — ответа нет);
- убираем префиксы вида «Ответ:»;
- берём первую непустую строку (модель часто дописывает пояснение с новой строки);
- срезаем кавычки, markdown-звёздочки и финальную точку.

In [ ]:
ANSWER_PREFIX = re.compile(r"^\s*(?:\*\*)?\s*(?:ответ|answer|итоговый ответ)\s*[:：\-—]\s*(?:\*\*)?", re.I)


def split_thinking(raw):
    # возвращает (размышления, ответ)
    if "</think>" in raw:
        thinking, answer = raw.rsplit("</think>", 1)
        return thinking.replace("<think>", "").strip(), answer
    if "<think>" in raw or raw.lstrip().startswith("<think>"):
        return raw.replace("<think>", "").strip(), ""   # размышление не закончилось - ответа нет
    return "", raw


def extract_answer(raw):
    _, answer = split_thinking(raw)
    answer = answer.strip()
    lines = [l for l in answer.splitlines() if l.strip()]
    answer = lines[0] if lines else ""
    answer = ANSWER_PREFIX.sub("", answer).strip()
    answer = answer.strip(" *_`\"'«»“”.;:").strip()
    return answer


assert extract_answer("<think>\nхм\n</think>\n\nОтвет: «Городская дума».\nПотому что...") == "Городская дума"
assert extract_answer("**Волга**") == "Волга"
assert extract_answer("<think>\nне успел") == ""

In [ ]:
RESULTS = {"baseline: frequent bigram": bigram_scores, "baseline: heuristic": heuristic_scores}


def run_and_score(name, prompts, max_new_tokens):
    t0 = time.time()
    raw, n_tok = generate(prompts, max_new_tokens, desc=name)
    preds = [extract_answer(r) for r in raw]
    scores, em, f1 = evaluate(preds)
    scores["sec"] = round(time.time() - t0, 1)
    RESULTS[name] = scores
    print(name, scores)
    df = pd.DataFrame({"question": data["question"], "gold": data["gold"], "bridge": data["bridge"],
                       "raw": raw, "pred": preds, "EM": em, "F1": f1, "gen_tokens": n_tok})
    return df

## Эксперимент 1 (2 балла): без thinking, без system prompt

Ответы в MultiQ короткие (1–4 слова), поэтому `max_new_tokens=48` — с запасом на случай, если модель начнёт отвечать предложением, но генерация не растягивается.

In [ ]:
MAX_NEW_TOKENS = 48

prompts_exp1 = [build_prompt(p, enable_thinking=False) for p in data["prompt"]]
exp1 = run_and_score("exp1: no system, no thinking", prompts_exp1, MAX_NEW_TOKENS)
exp1[["question", "gold", "pred", "EM", "F1"]]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


exp1: no system, no thinking 25/25
exp1: no system, no thinking {'EM': 0.12, 'F1_lemma': 0.24933333333333332, 'sec': 18.5}


,question,gold,pred,EM,F1
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Тумана,0.0,0.000000
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],"Глава государства, ставшего местом смерти Романа Николаевича Филипова, — это **Сирия",0.0,0.000000
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Гора Белуха,1.0,1.000000
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходревка впадает в реку Шаня,0.0,0.333333
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереванский городский совет,0.0,0.666667
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Ворговаш,0.0,0.000000
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],"Команда, в которой играет Карлос Де лос Кобос, — это «Керетаро",0.0,0.000000
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Ереван,1.0,1.000000
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Министр иностранных дел,0.0,0.000000
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Евгений Тефтелев,0.0,0.500000


### Разбор ошибок эксперимента 1

Раскладываем неточные ответы по типам — это и есть основа для system prompt во втором эксперименте:
- **пустой** ответ;
- **многословный**: все леммы правильного ответа есть в предсказании, но вокруг лишние слова (ответ предложением, пояснения) → recall 1, precision низкий;
- **ответ-мост**: модель назвала промежуточную сущность из Текста 1 (например, город рождения вместо органа власти этого города) — типичная ошибка multi-hop;
- **частично**: пересечение есть, но неполное (например, взяла только часть названия);
- **мимо**: ничего общего.

In [ ]:
def error_type(row):
    if row["EM"] == 1:
        return "верно (EM)"
    if not row["pred"]:
        return "пустой"
    pred_l = set(to_lemmas(row["pred"]))
    if any(set(to_lemmas(g)) <= pred_l for g in row["gold"]):
        return "многословный" if len(pred_l) > max(len(set(to_lemmas(g))) for g in row["gold"]) else "верно по леммам"
    if row["bridge"] and lemma_f1(row["pred"], [row["bridge"]]) >= 0.5:
        return "ответ-мост"
    if row["F1"] > 0:
        return "частично"
    return "мимо"


def error_report(df):
    df = df.copy()
    df["error_type"] = df.apply(error_type, axis=1)
    display(df["error_type"].value_counts().to_frame("n"))
    print("Средняя длина ответа в словах:", round(df["pred"].str.split().str.len().mean(), 2),
          "| у золота:", round(np.mean([len(g[0].split()) for g in df["gold"]]), 2))
    return df


exp1 = error_report(exp1)
exp1.loc[exp1["error_type"] != "верно (EM)", ["question", "gold", "bridge", "raw", "error_type"]]

,n
error_type,
мимо,14
частично,4
верно (EM),3
ответ-мост,2
многословный,1
верно по леммам,1


Средняя длина ответа в словах: 3.88 | у золота: 1.84


,question,gold,bridge,raw,error_type
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки,Тумана\n,мимо
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия,"Глава государства, ставшего местом смерти Романа Николаевича Филипова, — это **Сирия**.\n\n**Пояснение:**\nВ тексте ...",мимо
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев,Суходревка впадает в реку Шаня.\n,многословный
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван,Ереванский городский совет\n,частично
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,Ворговаш.\n,мимо
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро,"Команда, в которой играет Карлос Де лос Кобос, — это «Керетаро».\n",мимо
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия,Министр иностранных дел\n,мимо
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск,Евгений Тефтелев\n,частично
10,"Что за законодательный орган работает в городе, где родился Мовсес Геворкович Горгисян?",[городским советом],Ереван,Ереван\n,ответ-мост
11,На территории какой административно-территориальной единицы находится город рождения Антиповой Юлии Валерьевны?,[Зеленоградский административный округ],Зеленограде;Зеленоград,Город рождения Антиповой Юлии Валерьевны находится на территории Зеленограда.\n,мимо


## Эксперимент 2 (1 балл): + system prompt

В эксперименте 1 модель почти всегда отвечает **целым предложением**: пересказывает вопрос («Суходрев впадает в реку…», «Глава государства… — это Сирия»). Правильный ответ часто есть внутри, поэтому F1 по леммам ненулевой (≈0.25), а EM низкий (0.12). Отдельно встречаются **ответы-мосты**: модель называет промежуточную сущность вместо того, о чём спрашивают.

### Версия 1 — формат + явная двухшаговая схема

System prompt адресует обе ошибки:
- **многословность** → жёсткий формат: одна короткая подстрока из Текста 2, без предложения, пояснений и точки;
- **несовпадение формы** со золотом → копировать слова ровно в той форме, как в тексте;
- **ответ-мост** → описываем двухшаговую схему (найти мост в Тексте 1, затем ответ в Тексте 2) и говорим, что мост — не ответ;
- для числовых вопросов — только число.

Параметры генерации те же, что в эксперименте 1.

In [ ]:
SYSTEM_PROMPT_V1 = '''Ты отвечаешь на вопросы по двум текстам. Это вопрос "в два шага":
1) в Тексте 1 найди промежуточную сущность, о которой идет речь в вопросе (например, город, где родился человек);
2) в Тексте 2 найди про эту сущность то, что спрашивается в вопросе.
Промежуточная сущность из шага 1 почти никогда не является ответом.

Формат ответа:
- ответ - это короткий фрагмент (обычно 1-3 слова), скопированный из Текста 2 ровно в той форме, как он там написан;
- не пиши предложение, не повторяй вопрос, не добавляй пояснения, слово "Ответ", кавычки и точку в конце;
- если спрашивают год, дату или количество - напиши только число.

Пример.
Вопрос: На какой реке стоит город, где родился Иван Петров?
Текст 1: Иван Петров родился в Твери. Текст 2: Тверь расположена на реке Волге.
Правильный ответ: Волге'''

prompts_exp2 = [build_prompt(p, system_prompt=SYSTEM_PROMPT_V1, enable_thinking=False) for p in data["prompt"]]
exp2 = run_and_score("exp2a: system prompt v1, no thinking", prompts_exp2, MAX_NEW_TOKENS)
exp2 = error_report(exp2)
exp2[["question", "gold", "bridge", "pred", "EM", "F1", "error_type"]]

exp2a: system prompt v1, no thinking 25/25
exp2a: system prompt v1, no thinking {'EM': 0.04, 'F1_lemma': 0.10666666666666666, 'sec': 12.4}


,n
error_type,
мимо,13
ответ-мост,9
частично,1
верно (EM),1
верно по леммам,1


Средняя длина ответа в словах: 1.52 | у золота: 1.84


,question,gold,bridge,pred,EM,F1,error_type
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки,Вятка,0.0,0.000000,ответ-мост
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия,Сирия,0.0,0.000000,ответ-мост
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Алтая,4506,0.0,0.000000,мимо
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев,Суходрев,0.0,0.000000,ответ-мост
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван,Ереван,0.0,0.000000,ответ-мост
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,Ворговаш,0.0,0.000000,мимо
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро,Лигу,0.0,0.666667,частично
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Генрих,Ереван,1.0,1.000000,верно (EM)
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия,Министр иностранных дел,0.0,0.000000,мимо
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск,Борис Дубровский,0.0,0.000000,мимо


**Что получилось.** Формат починился — ответы стали короткими, — но качество упало: модель массово стала отвечать **мостом** или сущностью из самого вопроса («Вятка», «Сирия», «Ереван» — объекты, о которых написан Текст 2; «Ворговаш» — река из вопроса). Описание шага 1 («найди промежуточную сущность») маленькая модель восприняла как задание: она находит мост и на этом останавливается, а оговорка «мост не ответ» не срабатывает. Типичная история с промптами для маленьких моделей: перечисленное в инструкции действие модель скорее выполнит, чем запрет.

### Версия 2 — без описания шага с мостом

Итерируем по разбору ошибок версии 1:
- убираем пошаговую схему, в которой упоминается «промежуточная сущность»;
- формулируем через то, что модель видит: Текст 2 — статья о каком-то объекте, **сам объект ответом не является**, ответ — другое название из Текста 2;
- в примере показываем и неправильный ответ-мост, и правильный.

In [ ]:
SYSTEM_PROMPT_V2 = '''Ответь на вопрос по двум текстам одним коротким фрагментом из Текста 2.

Текст 2 описывает какой-то объект (город, реку, страну, человека). Сам этот объект НЕ является ответом.
Ответ - это другое название из Текста 2, о котором спрашивает вопрос: река, в которую впадает другая река; должность; орган власти; гора; город и т.п.

Пиши только ответ: 1-3 слова, скопированные из Текста 2 в той же форме. Без предложения, пояснений, кавычек и точки.

Пример.
Вопрос: На какой реке стоит город, где родился Иван Петров?
Текст 1: Иван Петров родился в Твери.
Текст 2: Тверь - город в России, расположен на реке Волге.
Неправильно: Тверь
Правильно: Волге'''

prompts_exp2b = [build_prompt(p, system_prompt=SYSTEM_PROMPT_V2, enable_thinking=False) for p in data["prompt"]]
exp2b = run_and_score("exp2b: system prompt v2, no thinking", prompts_exp2b, MAX_NEW_TOKENS)
exp2b = error_report(exp2b)
exp2b[["question", "gold", "bridge", "pred", "EM", "F1", "error_type"]]

exp2b: system prompt v2, no thinking 25/25
exp2b: system prompt v2, no thinking {'EM': 0.08, 'F1_lemma': 0.22133333333333333, 'sec': 12.8}


,n
error_type,
мимо,11
ответ-мост,6
частично,5
верно (EM),2
верно по леммам,1


Средняя длина ответа в словах: 2.08 | у золота: 1.84


,question,gold,bridge,pred,EM,F1,error_type
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки,Вятка,0.0,0.000000,ответ-мост
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия,Министр обороны,0.0,0.000000,мимо
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Алтая,Белуха,1.0,1.000000,верно (EM)
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев,Суходрев,0.0,0.000000,ответ-мост
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван,Ереванский городский совет,0.0,0.666667,частично
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,Ворговаш,0.0,0.000000,мимо
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро,Лига МХ,0.0,1.000000,верно по леммам
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Генрих,Ереван,1.0,1.000000,верно (EM)
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия,министр иностранных дел,0.0,0.000000,мимо
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск,Бывший ректор Челябинского государственного университета,0.0,0.000000,мимо


In [ ]:
# что поменялось относительно эксперимента 1
cmp = pd.DataFrame({"question": data["question"], "gold": data["gold"], "bridge": data["bridge"],
                    "exp1": exp1["pred"], "exp2a": exp2["pred"], "exp2b": exp2b["pred"],
                    "F1_exp1": exp1["F1"], "F1_exp2a": exp2["F1"], "F1_exp2b": exp2b["F1"]})
for v in ["exp2a", "exp2b"]:
    d = cmp[f"F1_{v}"] - cmp["F1_exp1"]
    print(f"{v} vs exp1: лучше {(d > 0).sum()} | хуже {(d < 0).sum()} | без изменений {(d == 0).sum()}")
err = pd.DataFrame({"exp1": exp1["error_type"].value_counts(), "exp2a": exp2["error_type"].value_counts(),
                    "exp2b": exp2b["error_type"].value_counts()}).fillna(0).astype(int)
display(err)
display(cmp)

# в эксперимент 3 берем лучшую версию system prompt
BEST_SYSTEM_PROMPT = SYSTEM_PROMPT_V2 if RESULTS["exp2b: system prompt v2, no thinking"]["F1_lemma"] >= \
    RESULTS["exp2a: system prompt v1, no thinking"]["F1_lemma"] else SYSTEM_PROMPT_V1
print("В эксперимент 3 идет:", "v2" if BEST_SYSTEM_PROMPT is SYSTEM_PROMPT_V2 else "v1")

exp2a vs exp1: лучше 1 | хуже 7 | без изменений 17
exp2b vs exp1: лучше 3 | хуже 4 | без изменений 18


,exp1,exp2a,exp2b
error_type,,,
верно (EM),3,1,2
верно по леммам,1,1,1
мимо,14,13,11
многословный,1,0,0
ответ-мост,2,9,6
частично,4,1,5


,question,gold,bridge,exp1,exp2a,exp2b,F1_exp1,F1_exp2a,F1_exp2b
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки,Тумана,Вятка,Вятка,0.000000,0.000000,0.000000
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия,"Глава государства, ставшего местом смерти Романа Николаевича Филипова, — это **Сирия",Сирия,Министр обороны,0.000000,0.000000,0.000000
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Алтая,Гора Белуха,4506,Белуха,1.000000,0.000000,1.000000
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев,Суходревка впадает в реку Шаня,Суходрев,Суходрев,0.333333,0.000000,0.000000
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван,Ереванский городский совет,Ереван,Ереванский городский совет,0.666667,0.000000,0.666667
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,Ворговаш,Ворговаш,Ворговаш,0.000000,0.000000,0.000000
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро,"Команда, в которой играет Карлос Де лос Кобос, — это «Керетаро",Лигу,Лига МХ,0.000000,0.666667,1.000000
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Генрих,Ереван,Ереван,Ереван,1.000000,1.000000,1.000000
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия,Министр иностранных дел,Министр иностранных дел,министр иностранных дел,0.000000,0.000000,0.000000
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск,Евгений Тефтелев,Борис Дубровский,Бывший ректор Челябинского государственного университета,0.500000,0.000000,0.000000


В эксперимент 3 идет: v2


## Эксперимент 3 (3 балла): thinking

Сложности и как они решаются:

**1. Отделение размышлений от ответа.** Чат-темплейт Qwen3.5 с `enable_thinking=True` заканчивает промпт открытым `<think>\n` (видно в выводе выше; без thinking темплейт подставляет пустой блок `<think>\n\n</think>`). Всё до последнего `</think>` — размышление, после — ответ. Это делают `split_thinking` / `extract_answer`; поэтому при декодировании не используем `skip_special_tokens`, а служебные токены вычищаем вручную, оставляя теги размышлений.

**2. Бюджет на размышления.** Общий бюджет = `THINK_TOKENS + ANSWER_TOKENS`, генерация в два этапа:
- **Этап 1:** генерируем не более `THINK_TOKENS` токенов с `stop_strings=["</think>"]` — генерация останавливается сразу, как только модель закрыла размышление.
- **Проверка:** есть ли `</think>` в сгенерированном.
  - есть → модель додумала; продолжаем генерацию с накопленного текста (промпт + размышление) ещё на `ANSWER_TOKENS`;
  - нет → обрубаем размышление: дописываем фразу-переход и `</think>` вручную и генерируем ответ с `ANSWER_TOKENS`.
- Так ответ гарантированно появляется, а суммарно на пример уходит не больше `THINK_TOKENS + ANSWER_TOKENS`.

**Выбор бюджета и параметров декодирования.** Что показали прогоны с жадной генерацией:
- с `THINK_TOKENS=512` модель не закончила думать ни в одном из 25 примеров (F1 0.16);
- с `THINK_TOKENS=1536` — тоже **0/25** (F1 0.18, 526 с). Причина видна в выводе размышлений: при жадном декодировании модель зацикливается и до конца бюджета повторяет одну и ту же строку («Constraint 5: The answer must be from Text 2», «Constraint 6: …», …). Увеличение бюджета такую петлю не лечит.

Разработчики Qwen прямо не рекомендуют жадное декодирование в режиме thinking именно из-за таких повторов и советуют сэмплирование (`temperature=0.6, top_p=0.95, top_k=20`). Добавляем к нему `repetition_penalty=1.1` против петель. Поэтому сравниваем три конфигурации:
1. greedy, бюджет 512 — исходный вариант;
2. sampling, бюджет 512;
3. sampling, бюджет 1536.

Для воспроизводимости перед каждым прогоном с сэмплированием фиксируем сид.

In [ ]:
ANSWER_TOKENS = MAX_NEW_TOKENS
THINK_SAMPLING = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20, repetition_penalty=1.1)
EXP3_CONFIGS = [("greedy", 512, None), ("sampling", 512, THINK_SAMPLING), ("sampling", 1536, THINK_SAMPLING)]
FORCE_STOP = "\n\nХорошо, времени на размышления больше нет, даю короткий итоговый ответ.\n</think>\n\n"


def generate_with_thinking(user_prompts, system_prompt=None, think_tokens=512, answer_tokens=ANSWER_TOKENS, gen_kwargs=None):
    prompts = [build_prompt(p, system_prompt=system_prompt, enable_thinking=True) for p in user_prompts]

    # этап 1: размышление с ограничением и остановкой на </think>
    stage1, think_n = generate(prompts, think_tokens, stop_strings=["</think>"], desc=f"thinking[{think_tokens}]",
                               gen_kwargs=gen_kwargs)

    # проверка: закончила ли модель думать
    finished = ["</think>" in s for s in stage1]
    continuations = []
    for prompt, s, ok in zip(prompts, stage1, finished):
        if ok:
            thinking = s[: s.index("</think>") + len("</think>")] + "\n\n"
        else:
            thinking = s.rstrip() + FORCE_STOP   # обрубаем размышление вручную
        continuations.append(prompt + thinking)

    # этап 2: ответ как продолжение накопленного текста
    stage2, answer_n = generate(continuations, answer_tokens, desc="answer", gen_kwargs=gen_kwargs)

    full = [c[len(p):] + a for c, p, a in zip(continuations, prompts, stage2)]
    return full, finished, think_n, answer_n


exp3_runs = {}
for decoding, budget, gen_kwargs in EXP3_CONFIGS:
    torch.manual_seed(SEED)
    t0 = time.time()
    raw, finished, think_n, answer_n = generate_with_thinking(data["prompt"].tolist(), BEST_SYSTEM_PROMPT,
                                                              budget, gen_kwargs=gen_kwargs)
    preds = [extract_answer(r) for r in raw]
    scores, em, f1 = evaluate(preds)
    scores["sec"] = round(time.time() - t0, 1)
    scores["finished"] = f"{sum(finished)}/{len(finished)}"
    name = f"exp3: {decoding}, budget {budget}"
    RESULTS[name] = scores
    exp3_runs[name] = pd.DataFrame({"question": data["question"], "gold": data["gold"], "bridge": data["bridge"],
                                    "raw": raw, "pred": preds, "EM": em, "F1": f1,
                                    "finished_thinking": finished, "think_tokens": think_n})
    print(name, scores)
    print(f"  закончила думать сама: {sum(finished)}/{len(finished)}; "
          f"токенов на размышление: медиана {int(np.median(think_n))}, макс {max(think_n)}; на ответ: макс {max(answer_n)}")

thinking[512] 25/25
answer 25/25
exp3: greedy, budget 512 {'EM': 0.04, 'F1_lemma': 0.16355555555555554, 'sec': 188.5, 'finished': '0/25'}
  закончила думать сама: 0/25; токенов на размышление: медиана 512, макс 512; на ответ: макс 48
thinking[512] 25/25
answer 25/25
exp3: sampling, budget 512 {'EM': 0.04, 'F1_lemma': 0.2397948717948718, 'sec': 194.5, 'finished': '0/25'}
  закончила думать сама: 0/25; токенов на размышление: медиана 512, макс 512; на ответ: макс 48
thinking[1536] 25/25
answer 25/25
exp3: sampling, budget 1536 {'EM': 0.08, 'F1_lemma': 0.2653333333333333, 'sec': 546.2, 'finished': '0/25'}
  закончила думать сама: 0/25; токенов на размышление: медиана 1536, макс 1536; на ответ: макс 22


In [ ]:
# детальный разбор для последней конфигурации
exp3 = exp3_runs[list(exp3_runs)[-1]]   # sampling, бюджет 1536
exp3["thinking"] = [split_thinking(r)[0] for r in exp3["raw"]]
exp3 = error_report(exp3)
for ok, label in [(True, "додумала сама"), (False, "обрубили")]:
    part = exp3[exp3.finished_thinking == ok]
    print(f"{label}: {len(part)} примеров, F1 = {part['F1'].mean():.3f}" if len(part) else f"{label}: 0 примеров")
exp3[["question", "gold", "pred", "EM", "F1", "finished_thinking", "think_tokens", "error_type"]]

,n
error_type,
мимо,10
ответ-мост,7
верно (EM),2
верно по леммам,2
частично,2
многословный,2


Средняя длина ответа в словах: 1.84 | у золота: 1.84
додумала сама: 0 примеров
обрубили: 25 примеров, F1 = 0.265


,question,gold,pred,EM,F1,finished_thinking,think_tokens,error_type
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Тумана,0.0,0.000000,False,1536,мимо
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Глава государства Сирии,0.0,0.000000,False,1536,ответ-мост
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Белуха,1.0,1.000000,False,1536,верно (EM)
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходревка,0.0,0.000000,False,1536,мимо
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереванское городское совете,0.0,1.000000,False,1536,верно по леммам
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,0.0,0.000000,False,1536,ответ-мост
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Лига МХ,0.0,1.000000,False,1536,верно по леммам
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Ереван,1.0,1.000000,False,1536,верно (EM)
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Глава государства,0.0,0.000000,False,1536,мимо
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Администрация,0.0,0.000000,False,1536,мимо


In [ ]:
# пример размышления целиком (лучший пример; по возможности - из тех, где модель додумала сама)
pool = exp3[exp3.finished_thinking] if exp3.finished_thinking.any() else exp3
i = int(pool["F1"].idxmax())
print("ВОПРОС:", data.loc[i, "question"], "| ЗОЛОТО:", data.loc[i, "gold"])
print(exp3.loc[i, "raw"])

ВОПРОС: Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун? | ЗОЛОТО: ['Белуха', 'гора Белуха']
Thinking Process:

1.  **Analyze the Request:**
    *   Input: Two texts (Text 1 and Text 2).
    *   Task: Answer a specific question based on the information in the two texts.
    *   Question: "Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?" (What is the highest point of a mountain range to which Holzun belongs?)
    *   Constraint 1: The answer must be based only on the information from Text 2 (not Text 1).
    *   Constraint 2: The answer must be short (1-3 words).
    *   Constraint 3: The format must be copied exactly from Text 2 without explanations, quotes, or periods.
    *   Constraint 4: The object being discussed is NOT the answer itself (the object described by the user text is not the answer). Wait, looking at the instruction again: "Ответ - это другое название из Текста 2" (The answer is another name from 

In [ ]:
# дополнительная конфигурация: sampling, бюджет 1536, БЕЗ system prompt
# в размышлениях выше видно, что модель находит ответ, но потом раз за разом перечитывает
# ограничения из system prompt ("Wait, the instruction says...") и не закрывает </think>.
# проверяем, закончит ли она думать, если system prompt убрать
name = "exp3: sampling, budget 1536, no system"
torch.manual_seed(SEED)
t0 = time.time()
raw_ns, fin_ns, th_ns, an_ns = generate_with_thinking(data["prompt"].tolist(), None, 1536, gen_kwargs=THINK_SAMPLING)
preds_ns = [extract_answer(r) for r in raw_ns]
scores_ns, em_ns, f1_ns = evaluate(preds_ns)
scores_ns["sec"] = round(time.time() - t0, 1)
scores_ns["finished"] = f"{sum(fin_ns)}/{len(fin_ns)}"
RESULTS[name] = scores_ns
exp3_runs[name] = pd.DataFrame({"question": data["question"], "gold": data["gold"], "bridge": data["bridge"],
                                "raw": raw_ns, "pred": preds_ns, "EM": em_ns, "F1": f1_ns,
                                "finished_thinking": fin_ns, "think_tokens": th_ns})
print(name, scores_ns)
print(f"  закончила думать сама: {sum(fin_ns)}/{len(fin_ns)}; токенов на размышление: медиана {int(np.median(th_ns))}, мин {min(th_ns)}")
if any(fin_ns):
    done = [f for f, ok in zip(f1_ns, fin_ns) if ok]
    cut = [f for f, ok in zip(f1_ns, fin_ns) if not ok]
    print(f"  F1 среди додумавших: {np.mean(done):.3f}; среди обрубленных: {np.mean(cut) if cut else float('nan'):.3f}")


thinking[1536] 25/25
answer 25/25
exp3: sampling, budget 1536, no system {'EM': 0.12, 'F1_lemma': 0.348, 'sec': 540.1, 'finished': '0/25'}
  закончила думать сама: 0/25; токенов на размышление: медиана 1536, мин 1536


## Итоги

In [ ]:
summary = pd.DataFrame(RESULTS).T.reindex(columns=["EM", "F1_lemma", "sec", "finished"])
summary[["EM", "F1_lemma"]] = summary[["EM", "F1_lemma"]].astype(float).round(3)
summary

,EM,F1_lemma,sec,finished
baseline: frequent bigram,0.04,0.040,NaN,NaN
baseline: heuristic,0.20,0.200,NaN,NaN
"exp1: no system, no thinking",0.12,0.249,18.5,NaN
"exp2a: system prompt v1, no thinking",0.04,0.107,12.4,NaN
"exp2b: system prompt v2, no thinking",0.08,0.221,12.8,NaN
"exp3: greedy, budget 512",0.04,0.164,188.5,0/25
"exp3: sampling, budget 512",0.04,0.240,194.5,0/25
"exp3: sampling, budget 1536",0.08,0.265,546.2,0/25
"exp3: sampling, budget 1536, no system",0.12,0.348,540.1,0/25


In [ ]:
# сводная таблица ответов по всем экспериментам
pd.DataFrame({"question": data["question"], "gold": data["gold"], "bridge": data["bridge"],
              "heuristic": heuristic_preds, "exp1": exp1["pred"], "exp2a": exp2["pred"], "exp2b": exp2b["pred"],
              **{name.replace("exp3: ", "exp3 "): run["pred"] for name, run in exp3_runs.items()}})

,question,gold,bridge,heuristic,exp1,exp2a,exp2b,"exp3 greedy, budget 512","exp3 sampling, budget 512","exp3 sampling, budget 1536","exp3 sampling, budget 1536, no system"
0,"В какую реку впадает река, притоком которой является Мурдюг?","[Каму, Камы]",Вятки,Европейской,Тумана,Вятка,Вятка,Тумана,Вятка,Тумана,Мурдюг впадает в Вятку
1,"Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?",[президентов],Сирия,Сурийя,"Глава государства, ставшего местом смерти Романа Николаевича Филипова, — это **Сирия",Сирия,Министр обороны,Глава государства,глава государства,Глава государства Сирии,"Глава государства, ставшее место смерти Романа Николаевича Филипова, занимал должность заместителя командира эскадри..."
2,"Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?","[Белуха, гора Белуха]",Алтая,Туулу Алтай,Гора Белуха,4506,Белуха,Горная система,Гора Белуха (4506 м),Белуха,Белуха
3,"В какую реку впадает река, притоком которой является Суходревка?",[Шаня],Суходрев,,Суходревка впадает в реку Шаня,Суходрев,Суходрев,Впадает в реку Шаня,Впадает в реку Суходрев,Суходревка,Суходревка впадает в реку Суходрев
4,"Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?",[ереванским городским советом],Ереван,,Ереванский городский совет,Ереван,Ереванский городский совет,Ереванский городский совет,Ереванское городское совете,Ереванское городское совете,Ереванское городское совете
5,"Как называется река, питающаяся от реки, притоком которой является Ворговаш?",[Сабля],Кисьма,Светишонка,Ворговаш,Ворговаш,Ворговаш,Ворговаш,Ворговаш,Кисьма,Ворговаш
6,"В какой лиге играет команда, тренера которой зовут Карлос Де лос Кобос?",[Лиге МХ],Керетаро,Лиге МХ,"Команда, в которой играет Карлос Де лос Кобос, — это «Керетаро",Лигу,Лига МХ,Лига МХ,лига,Лига МХ,Лига МХ
7,"Как называется населенный пункт, в котором родился сын Гамлета Апетнаковича Мхитаряна?",[Ереван],Генрих,Францию,Ереван,Ереван,Ереван,Ереван,Ереван,Ереван,Ереван
8,"Какая должность у главы государства в стране, чьим гражданином был Мухаммед Ашави?",[президентов],Сирия,Сурийя,Министр иностранных дел,Министр иностранных дел,министр иностранных дел,министр иностранных дел,,Глава государства,Министр иностранных дел
9,"Кто является главой администрации в городе, где родилась Диана Александровна Циринг?",[Евгению Тефтелеву],Челябинск,Евгению Тефтелеву,Евгений Тефтелев,Борис Дубровский,Бывший ректор Челябинского государственного университета,Глава администрации,"Город, где родилась Диана Александровна Циринг, Челябинск",Администрация,Евгений Тефтелев


### Выводы

| | EM | F1 по леммам | время, с | додумала сама |
|---|---|---|---|---|
| частотный биграмм | 0.04 | 0.04 | — | — |
| эвристический бейзлайн | 0.20 | 0.20 | — | — |
| exp1: без system, без thinking | 0.12 | 0.25 | 18 | — |
| exp2a: system prompt v1 | 0.04 | 0.11 | 12 | — |
| exp2b: system prompt v2 | 0.08 | 0.22 | 13 | — |
| exp3: thinking, greedy, 512 | 0.04 | 0.16 | 189 | 0/25 |
| exp3: thinking, sampling, 512 | 0.04 | 0.24 | 195 | 0/25 |
| exp3: thinking, sampling, 1536 | 0.08 | 0.27 | 546 | 0/25 |
| exp3: thinking, sampling, 1536, без system prompt | **0.12** | **0.35** | 540 | 0/25 |

- **Метрики.** EM для MultiQ очень строгая: золото — спан из текста в косвенном падеже («Камы», «президентов»), и любой ответ в другой форме или с лишними словами получает 0. F1 по леммам снимает проблему формы и частично засчитывает многословные ответы. Разрыв F1 − EM в exp1 (0.12 → 0.25) показывает, сколько верных по сути ответов «утонуло» в предложении.
- **Бейзлайны.** Частотный биграмм почти не попадает: самые частотные пары — повторяющееся название темы текста. Эвристика «предложение Текста 2, ближайшее к вопросу → сущность, которой нет в вопросе, Тексте 1 и заголовке Текста 2» использует структуру multi-hop вопроса и даёт EM 0.20 — выше EM модели во всех конфигурациях. Для 0.8B-модели это сильный ориентир.
- **Exp1.** Модель понимает вопрос, но отвечает предложением с пересказом вопроса, поэтому EM низкий при умеренном F1.
- **Exp2.** Первая версия system prompt починила формат, но спровоцировала ответ-мост: модель выполнила описанный «шаг 1» (найти промежуточную сущность) и на нём остановилась — F1 упал с 0.25 до 0.11. Вторая версия, где задача описана через результат («объект Текста 2 — не ответ») и в примере показан неправильный ответ-мост, вернула F1 до 0.22 при коротком формате. До exp1 по F1 она не дотягивает: короткий ответ работает по принципу «всё или ничего», а длинный ответ exp1 часто случайно содержит нужное слово. Вывод: маленькой модели инструкции лучше давать через желаемый результат, а не через промежуточные шаги, и показывать в примере неправильный ответ.
- **Exp3, механика.** Двухэтапная генерация работает: на этапе 1 размышление ограничено бюджетом и `stop_strings=["</think>"]`, если тег не появился — размышление обрубается вручную, и на этапе 2 модель всегда выдаёт ответ (максимум 48 токенов), а постобработка отделяет его от размышления.
- **Exp3, что мешает модели закончить думать.** Ни в одной конфигурации Qwen3.5-0.8B не закрыла `</think>` сама, даже за 1536 токенов. Разбор размышлений показывает две причины: (1) при жадном декодировании модель зацикливается и повторяет одну строку до конца бюджета («Constraint 5: …», «Constraint 6: …») — сэмплирование с `repetition_penalty` это лечит и поднимает F1 с 0.16 до 0.24 при том же бюджете; (2) с system prompt модель находит ответ, но снова и снова перепроверяет его на соответствие ограничениям («Wait, the instruction says…»). Без system prompt размышление становится содержательнее, и это лучшая конфигурация всего задания: EM 0.12, F1 0.35, хотя модель по-прежнему не укладывается в 1536 токенов.
- **Цена.** Даже обрубленное размышление помогает: модель успевает связать два текста через мост, и вынужденный короткий ответ после `</think>` оказывается точнее, чем ответ без размышлений (F1 0.35 против 0.25). Но это в 30 раз дороже по времени (540 с против 18 с). Для такой маленькой модели и экстрактивной задачи разумный компромисс — сэмплирование, без system prompt, с бюджетом, обрезаемым принудительно.